In [1]:
import pandas as pd
import numpy as np
import json

species_list = ['Abies_alba', 'Acer_campestre', 'Acer_pseudoplatanus', 'Alnus_glutinosa', 'Alnus_incana', 'Betula_pendula', 'Betula_pubescens', 'Carpinus_betulus', 'Corylus_avellana', 'Fagus_sylvatica', 'Fraxinus_excelsior', 'Picea_abies', 'Pinus_halepensis', 'Pinus_nigra', 'Pinus_pinaster', 'Pinus_sylvestris', 'Populus_tremula', 'Prunus_avium', 'Quercus_ilex', 'Quercus_petraea', 'Quercus_pubescens', 'Quercus_robur', 'Quercus_suber', 'Sorbus_aucuparia']

# bands start at 1, not 0!
band_names = {1: 'historical', 2: 'present', 5: 'ssp1', 6: 'ssp1a', 9: 'ssp2', 10: 'ssp2a', 13: 'ssp5', 14: 'ssp5a'}
scenarios = list(band_names.values())
# starting at 1, because we want to include "present": the optimized portfolios should already work today.
opt_scenarios = list(band_names.values())[1:]


CLIMATE_DATA_RES = 0.0083333

import amoc_computations as amoc_comp
import amoc_plots
import amoc_optimization
import importlib
importlib.reload(amoc_plots)
importlib.reload(amoc_comp)

ERROR 1: PROJ: proj_create_from_database: Open of /home/konni/anaconda3/envs/amoc/share/proj failed


<module 'amoc_computations' from '/home/konni/Documents/phd/amoc_forestry/amoc_computations.py'>

In [2]:
df_final_from_disk = pd.read_csv('data/all_data_newamoc_with_present_and_historical_every_100.csv', sep=';').set_index(['y', 'x'])

In [3]:
df_final = df_final_from_disk.copy()

In [4]:
df_final['opt_single_species'] = 0
df_final['opt_single_species_min_prob'] = 0.0
df_final['opt_portfolio'] = None
df_final['opt_portfolio'] = df_final['opt_portfolio'].astype(object)
df_final['opt_portfolio_min_prob'] = 0.0

# Run optimization in parallel

In [5]:
# takes 5 minutes when using every 100th entry, 10 cores

from multiprocessing import get_context

importlib.reload(amoc_optimization)

ctx = get_context("spawn")

tasks = [
    (idx, opt_scenarios, species_list, row)
    for idx, row in df_final.iterrows()
]


with ctx.Pool(10) as pool:
    results = pool.map(amoc_optimization.run_optimization_and_combine_with_row_index, tasks)

In [7]:
#
for row_idx, result in results:
    amoc_optimization.set_results_into_dataframe(df_final, opt_scenarios, species_list, result, row_idx)

In [8]:
# takes 15s
df_final['opt_portfolio'] = df_final['opt_portfolio'].apply(json.dumps)
df_final.to_csv("data/optimized.csv", sep=';')

In [9]:
df_final.head()

,,Abies_alba-historical,Abies_alba-present,Abies_alba-ssp1,Abies_alba-ssp2,Abies_alba-ssp5,Abies_alba-ssp1a,Abies_alba-ssp2a,Abies_alba-ssp5a,Acer_campestre-historical,Acer_campestre-present,...,opt_portfolio_max_prob,opt_single_species_prob_present,opt_single_species_prob_ssp1,opt_single_species_prob_ssp1a,opt_single_species_prob_ssp2,opt_single_species_prob_ssp2a,opt_single_species_prob_ssp5,opt_single_species_prob_ssp5a,opt_single_species_avg_prob,opt_single_species_max_prob
y,x,,,,,,,,,,,,,,,,,,,,,
71.104027,25.554027,1.0,2.0,2.0,1.0,8.0,0.0,1.0,12.0,0.0,0.0,...,33.868065,13.0,8.0,40.0,5.0,55.0,3.0,56.0,25.714286,56.0
71.087360,25.570694,0.0,3.0,6.0,3.0,10.0,0.0,1.0,17.0,0.0,0.0,...,40.075416,14.0,9.0,46.0,6.0,62.0,3.0,68.0,29.714286,68.0
71.070694,25.320694,2.0,2.0,1.0,0.0,5.0,0.0,1.0,11.0,0.0,0.0,...,28.409485,12.0,6.0,35.0,4.0,45.0,2.0,40.0,20.571429,45.0
71.062360,25.520694,0.0,1.0,4.0,4.0,8.0,0.0,0.0,6.0,0.0,0.0,...,27.446910,7.0,7.0,16.0,4.0,27.0,2.0,39.0,14.571429,39.0
71.054027,25.579027,0.0,1.0,5.0,6.0,10.0,0.0,0.0,6.0,0.0,0.0,...,32.399324,8.0,8.0,17.0,5.0,31.0,3.0,46.0,16.857143,46.0
